In [1]:
from gensim.models import LdaModel
import json
import time
import pandas as pd

from nltk.tokenize import word_tokenize
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from itertools import combinations

In [2]:
K_RANGE = list(range(3, 20))
TOP_N = 10
RANDOM_STATE = 1618 # same random state as default for MATAVE topic model

In [3]:
try:
    with open('../dataProcessed/nurseNotesProcessed.json', 'r') as file:
        nurse_notes = json.load(file)
    print("File loaded successfully.")
    
except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")

File loaded successfully.


In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, TOP_N=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:TOP_N]).intersection(set(topic2[:TOP_N])))
        redundancy = overlap / TOP_N
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
def lda_analysis(texts, dataset_name):
    # Prepare components for evaluation.
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)
    corpus = [dictionary.doc2bow(text) for text in tokenized_texts]

    metric_results = {'Dataset Name': [], 'Algorithm Name': [], 'Coherence': [], 'Diversity': [], 'Redundancy': [], 'Time': [], 'Top Topic Words': []}
    
    for k in K_RANGE:
        start = time.time()

        lda_model = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=k,
            random_state=RANDOM_STATE,
            passes=10
        )

        lda_topics = [
            [word for word, _ in lda_model.show_topic(i, topn=TOP_N)]
            for i in range(k)
        ]

        end = time.time()

        metric_results['Algorithm Name'].append(f'LDA (K={k})')
        metric_results['Dataset Name'].append(dataset_name)
        metric_results['Coherence'].append(
            get_coherence_score(lda_topics, tokenized_texts, dictionary, 'c_v')
        )
        metric_results['Diversity'].append(get_diversity_score(lda_topics))
        metric_results['Redundancy'].append(compute_topic_redundancy(lda_topics))
        metric_results['Time'].append(end - start)
        metric_results['Top Topic Words'].append(lda_topics)
    
    return metric_results

In [6]:
all_results = []
all_texts = []
for key in nurse_notes:
    all_results.append(pd.DataFrame(lda_analysis(nurse_notes[key], key)))
    all_texts.extend(nurse_notes[key])
combined_df = pd.concat(all_results)

In [7]:
def get_top_topics(temp_df):
    metrics = ['Coherence', 'Diversity', 'Redundancy']

    for metric in metrics:
        print(f"-------{metric}-------")
        temp_top_row = temp_df.loc[temp_df[metric].idxmax()]['Top Topic Words']
        print(f"Number of Topics: {len(temp_top_row)}")
        for topic in temp_top_row:
            # print(' '.join(topic))
            print(topic)
        print("\n")

In [8]:
for patient in list(nurse_notes.keys()):
    print("----------------------------")
    print(patient)
    print("----------------------------")
    get_top_topics(combined_df[combined_df['Dataset Name'] == patient])

----------------------------
P1
----------------------------
-------Coherence-------
Number of Topics: 3
['resident', 'staff', 'care', 'assist', 'night', 'morning', 'good', 'content', 'plan', 'take']
['check', 'resident', 'self', 'asleep', 'ongoing', 'comfortable', 'sleep', 'safety', 'settle', 'concern']
['resident', 'chart', 'care', 'med', 'need', 'appear', 'form', 'give', 'assist', 'good']


-------Diversity-------
Number of Topics: 3
['resident', 'staff', 'care', 'assist', 'night', 'morning', 'good', 'content', 'plan', 'take']
['check', 'resident', 'self', 'asleep', 'ongoing', 'comfortable', 'sleep', 'safety', 'settle', 'concern']
['resident', 'chart', 'care', 'med', 'need', 'appear', 'form', 'give', 'assist', 'good']


-------Redundancy-------
Number of Topics: 7
['morning', 'staff', 'breakfast', 'report', 'good', 'adls', 'intake', 'assist', 'concern', 'room']
['resident', 'check', 'asleep', 'ongoing', 'self', 'toilette', 'comfortable', 'peaceful', 'care', 'change']
['check', 'resi

In [9]:
all_texts_df = pd.DataFrame(lda_analysis(all_texts, "All Texts"))

In [10]:
get_top_topics(all_texts_df)

-------Coherence-------
Number of Topics: 13
['prescribe', 'baseline', 'nil', 'wash', 'assist', 'med', 'resident', 'take', 'meal', 'skin']
['resident', 'bright', 'care', 'take', 'nil', 'alert', 'concern', 'assist', 'appear', 'medication']
['sleep', 'bed', 'nocte', 'resident', 'continue', 'early', 'administer', 'medication', 'check', 'overnight']
['pain', 'prn', 'request', 'right', 'resident', 'inform', 'give', 'gp', 'paracetamol', 'eating']
['resident', 'note', 'time', 'low', 'receive', 'room', 'fall', 'till', 'observe', 'daughter']
['night', 'resident', 'settle', 'sleep', 'medication', 'give', 'check', 'staff', 'continue', 'voice']
['complaint', 'chart', 'resident', 'voice', 'nil', 'form', 'need', 'appear', 'care', 'attend']
['care', 'form', 'good', 'resident', 'chart', 'med', 'appear', 'assist', 'attend', 'give']
['resident', 'med', 'good', 'chart', 'concern', 'appear', 'assist', 'give', 'form', 'nil']
['sensor', 'place', 'mat', 'safety', 'check', 'bed', 'appear', 'toilet', 'alarm', 